In [1]:
import sys, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge

sys.path.insert(0, "..")
from src.forecasting_utils import all_metrics

import warnings; warnings.filterwarnings("ignore")

In [2]:
ml_val   = pd.read_parquet("../artifacts/ml_val_predictions.parquet")
ml_test  = pd.read_parquet("../artifacts/ml_test_predictions.parquet")
lstm_val  = pd.read_parquet("../artifacts/lstm_val_predictions.parquet")[["id", "date", "pred_lstm"]]
lstm_test = pd.read_parquet("../artifacts/lstm_test_predictions.parquet")[["id", "date", "pred_lstm"]]

print("ML val :", ml_val.shape,  "| LSTM val :", lstm_val.shape)
print("ML test:", ml_test.shape, "| LSTM test:", lstm_test.shape)

ML val : (30120, 6) | LSTM val : (30120, 3)
ML test: (27610, 10) | LSTM test: (27610, 3)


In [3]:
val  = ml_val.merge(lstm_val,  on=["id", "date"], how="inner")
test = ml_test.merge(lstm_test, on=["id", "date"], how="inner")

print(f"Aligned val rows:  {len(val):,}")
print(f"Aligned test rows: {len(test):,}")
val.head()

Aligned val rows:  30,120
Aligned test rows: 27,610


,id,date,sales,pred_lgbm,pred_xgb,pred_rf,pred_lstm
0,FOODS_1_002_TX_2_validation,2016-01-01,0,0.263521,0.276048,0.175657,0.040272
1,FOODS_1_002_TX_2_validation,2016-01-02,0,0.291569,0.306552,0.257988,0.063496
2,FOODS_1_002_TX_2_validation,2016-01-03,1,0.243170,0.265247,0.217176,0.108030
3,FOODS_1_002_TX_2_validation,2016-01-04,0,0.231514,0.233321,0.250068,0.089626
4,FOODS_1_002_TX_2_validation,2016-01-05,0,0.225840,0.229192,0.237589,0.071417


In [4]:
# Per methodology §3.6.1 Tier 4: LightGBM + XGBoost + RandomForest → Ridge meta-learner
TREE_BASE = ["pred_lgbm", "pred_xgb", "pred_rf"]

X_meta_val  = val[TREE_BASE].values
y_meta_val  = val["sales"].values
X_meta_test = test[TREE_BASE].values
y_meta_test = test["sales"].values

# Ridge with positive weights (forecast combination weights should not be negative)
# and intercept to absorb any systematic bias
ridge_trees = Ridge(alpha=1.0, positive=True, fit_intercept=True, random_state=42)
ridge_trees.fit(X_meta_val, y_meta_val)

print("Ridge weights (trees-only):")
for name, w in zip(TREE_BASE, ridge_trees.coef_):
    print(f"  {name:12s} → {w:.4f}")
print(f"  intercept    → {ridge_trees.intercept_:.4f}")

Ridge weights (trees-only):
  pred_lgbm    → 0.0000
  pred_xgb     → 0.7542
  pred_rf      → 0.3996
  intercept    → -0.1287


In [5]:
ALL_BASE = ["pred_lgbm", "pred_xgb", "pred_rf", "pred_lstm"]

ridge_all = Ridge(alpha=1.0, positive=True, fit_intercept=True, random_state=42)
ridge_all.fit(val[ALL_BASE].values, val["sales"].values)

print("Ridge weights (trees + LSTM):")
for name, w in zip(ALL_BASE, ridge_all.coef_):
    print(f"  {name:12s} → {w:.4f}")
print(f"  intercept    → {ridge_all.intercept_:.4f}")

Ridge weights (trees + LSTM):
  pred_lgbm    → 0.1402
  pred_xgb     → 0.4010
  pred_rf      → 0.3312
  pred_lstm    → 0.3341
  intercept    → -0.0961


In [6]:
ensemble_trees_test = np.clip(ridge_trees.predict(X_meta_test), 0, None)
ensemble_all_test   = np.clip(ridge_all.predict(test[ALL_BASE].values), 0, None)

results = []
# Individual base models for context
for col, label in [("pred_lgbm", "LightGBM"), ("pred_xgb", "XGBoost"),
                   ("pred_rf",  "RandomForest"), ("pred_lstm", "LSTM")]:
    results.append(all_metrics(y_meta_test, test[col].values, label))

# Simple average (a useful sanity-check ensemble)
simple_avg = test[TREE_BASE].mean(axis=1).values
results.append(all_metrics(y_meta_test, simple_avg, "SimpleAvg_trees"))

# The methodology ensemble + extended ensemble
results.append(all_metrics(y_meta_test, ensemble_trees_test, "STACK_Ridge_trees"))
results.append(all_metrics(y_meta_test, ensemble_all_test,   "STACK_Ridge_all"))

results_df = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)
results_df.to_csv("../data/processed/ensemble_results.csv", index=False)
display(results_df)

,model,MAE,RMSE,MAPE_pct,WMAPE_pct,Pred10_pct
0,STACK_Ridge_trees,0.952343,1.859784,57.823639,73.583102,9.621816
1,SimpleAvg_trees,0.969972,1.899602,53.848966,74.945238,9.355973
2,XGBoost,0.970340,1.931625,53.150087,74.973658,9.355973
3,LightGBM,0.976350,1.952907,53.663774,75.438034,9.218763
4,RandomForest,0.977026,1.879122,55.597250,75.490264,9.870509
5,STACK_Ridge_all,1.012635,1.837998,54.193830,78.241591,11.096818
6,LSTM,1.208860,2.363520,58.493775,93.402975,7.786639


In [7]:
test_out = test.copy()
test_out["pred_ensemble_trees"] = ensemble_trees_test
test_out["pred_ensemble_all"]   = ensemble_all_test
test_out.to_parquet("../artifacts/ensemble_test_predictions.parquet", index=False)
joblib.dump(ridge_trees, "../models/stacking_meta_ridge_trees.pkl")
joblib.dump(ridge_all,   "../models/stacking_meta_ridge_all.pkl")
print("Saved ensemble model + predictions.")

Saved ensemble model + predictions.


In [9]:
# Pull in baseline + ensemble + ml results into one consolidated view
baselines = pd.read_csv("../data/processed/baseline_results_full.csv")
all_models = pd.concat([baselines, results_df], ignore_index=True).sort_values("MAE")
all_models.to_csv("../data/processed/all_model_results.csv", index=False)
display(all_models)

# Pick best ensemble by MAE (operationally meaningful, aligned with Ridge's loss)
best_classical = baselines.sort_values("MAE").iloc[0]
best_ml        = results_df[results_df["model"].isin(["LightGBM","XGBoost","RandomForest","LSTM"])].sort_values("MAE").iloc[0]
best_ensemble  = results_df[results_df["model"].str.startswith("STACK_")].sort_values("MAE").iloc[0]

def improvement(base, new): return (base - new) / base * 100

print(f"\nBest classical : {best_classical['model']:20s} MAE={best_classical['MAE']:.3f}  RMSE={best_classical['RMSE']:.3f}  MAPE={best_classical['MAPE_pct']:.1f}%")
print(f"Best ML        : {best_ml['model']:20s} MAE={best_ml['MAE']:.3f}  RMSE={best_ml['RMSE']:.3f}  MAPE={best_ml['MAPE_pct']:.1f}%")
print(f"Best ensemble  : {best_ensemble['model']:20s} MAE={best_ensemble['MAE']:.3f}  RMSE={best_ensemble['RMSE']:.3f}  MAPE={best_ensemble['MAPE_pct']:.1f}%")
print(f"\nEnsemble vs best classical (Chapter 4 headline numbers):")
print(f"  MAE improvement : {improvement(best_classical['MAE'],      best_ensemble['MAE']):+.1f}%")
print(f"  RMSE improvement: {improvement(best_classical['RMSE'],     best_ensemble['RMSE']):+.1f}%")
print(f"  MAPE improvement: {improvement(best_classical['MAPE_pct'], best_ensemble['MAPE_pct']):+.1f}%")
print(f"  WMAPE improvement: {improvement(best_classical['WMAPE_pct'], best_ensemble['WMAPE_pct']):+.1f}%")

,model,MAE,RMSE,MAPE_pct,WMAPE_pct,Pred10_pct
5,STACK_Ridge_trees,0.952343,1.859784,57.823639,73.583102,9.621816
6,SimpleAvg_trees,0.969972,1.899602,53.848966,74.945238,9.355973
7,XGBoost,0.970340,1.931625,53.150087,74.973658,9.355973
8,LightGBM,0.976350,1.952907,53.663774,75.438034,9.218763
9,RandomForest,0.977026,1.879122,55.597250,75.490264,9.870509
1,ARIMA,0.997096,2.054741,58.447490,77.041001,8.532716
10,STACK_Ridge_all,1.012635,1.837998,54.193830,78.241591,11.096818
3,Croston_SBA,1.020811,2.088825,57.634707,78.873321,7.417889
2,Croston,1.033128,2.082374,58.107241,79.825023,8.850013
0,MA,1.080141,2.554272,64.361831,83.457451,5.548409



Best classical : ARIMA                MAE=0.997  RMSE=2.055  MAPE=58.4%
Best ML        : XGBoost              MAE=0.970  RMSE=1.932  MAPE=53.2%
Best ensemble  : STACK_Ridge_trees    MAE=0.952  RMSE=1.860  MAPE=57.8%

Ensemble vs best classical (Chapter 4 headline numbers):
  MAE improvement : +4.5%
  RMSE improvement: +9.5%
  MAPE improvement: +1.1%
  WMAPE improvement: +4.5%
